# Sistema basado en reglas para la seguridad de edificios
Se desea desarrollar un sistema basado en reglas que permita detectar posibles intrusiones en los edificios de un campus universitario y ejecutar las acciones de seguridad correspondientes. El sistema recibe información de distintos sensores instalados en los edificios y debe actuar de acuerdo con las siguientes condiciones:
* Si se detecta movimiento en un edificio durante el horario nocturno, el sistema considera que existe actividad sospechosa.
* Si existe actividad sospechosa en un edificio y alguna de sus puertas está abierta, el sistema considera que se ha producido una posible intrusión.
* Cuando se detecta una posible intrusión, el sistema debe registrar una alerta de seguridad y ejecutar la acción de activar la alarma del edificio.
* Cuando existe una alerta de seguridad en un edificio, el sistema debe ejecutar la acción de notificar al vigilante correspondiente.

Inicialmente, el sistema dispone de la siguiente información:
* Edificio A: se ha detectado movimiento, es horario nocturno y una puerta está abierta.
* Edificio B: se ha detectado movimiento y es horario nocturno. No se dispone de información sobre el estado de sus puertas.
* Edificio C: se ha detectado movimiento, es horario diurno y una puerta está abierta.

In [ ]:
import sys
!{sys.executable} -m pip install clipspy

In [ ]:
import clips
import logging

def crear_entorno():
    """
    Inicializa un nuevo entorno de trabajo independiente en CLIPS, configura el sistema
    de logs de Python y redirige las salidas por pantalla internas de CLIPS (salidas estándar y de error)
    hacia el logger de Python.
    
    Salida: env (clips.Environment): El objeto del entorno de CLIPS listo para ser usado.
    """
    env = clips.Environment()
    logging.basicConfig(level = 10, format = '%(message)s')
    router = clips.LoggingRouter()
    env.add_router(router)
    return env
     
def print_facts():
    """
    Imprime todos los hechos (facts) almacenados actualmente en la memoria de trabajo (Working Memory).
    Para cada hecho, muestra su índice numérico asignado por CLIPS (f.index) y su contenido/estructura.
    """
    for f in env.facts():
        print(f.index, f)
        
def print_rules():
    """
    Inspecciona y muestra en pantalla todas las reglas (defrules) que han sido 
    cargadas y compilaron con éxito en el entorno de CLIPS.
    """
    for r in env.rules():
        print(r)
        
def print_agenda():
    """
    Muestra el estado actual de la Agenda de CLIPS.
    La Agenda contiene las 'activaciones', es decir, aquellas reglas cuyas condiciones 
    (lado izquierdo / LHS) se han satisfecho con los hechos actuales de la memoria de trabajo 
    y que están a la espera de ser ejecutadas al llamar a env.run().
    """
    # env.activations() devuelve la lista de reglas listas para dispararse (junto con los hechos que las activaron)
    for a in env.activations():
        print(a)


"""
Instancia el entorno principal de CLIPS invocando la función de configuración.
A partir de este punto, 'env' actúa como el contenedor global de hechos y reglas.
"""
env = crear_entorno()

"""
Registra las funciones de Python ('print_facts', 'print_rules', 'print_agenda') dentro del entorno de CLIPS. 
Esto crea un puente que permite invocar estas funciones directamente desde el código nativo de CLIPS 
(por ejemplo, escribiendo '(print_facts)' dentro del consecuente/RHS de una regla).
"""
env.define_function(print_facts)
env.define_function(print_rules)
env.define_function(print_agenda)

In [ ]:
facts = """
(deffacts hechos-iniciales
   (movimiento A)
   (horario A nocturno)
   (puerta A abierta)

   (movimiento B)
   (horario B nocturno)

   (movimiento C)
   (horario C diurno)
   (puerta C abierta)
)
"""

"""
Compila y registra la definición de los hechos iniciales (el bloque 'deffacts' guardado en la variable 'facts') dentro del entorno de CLIPS. 
Esto solo guarda la definición en la memoria del motor; los hechos todavía no existen en la memoria de trabajo.
"""
env.build(facts)

"""
Reinicia el estado del motor de inferencia: vacía cualquier hecho previo de la memoria de trabajo,
inserta el hecho inicial por defecto '(initial-fact)', y vuelca en la memoria de trabajo todos los hechos 
definidos previamente dentro de los bloques 'deffacts' 
"""
env.reset()

print_facts()


In [ ]:
r1 = """(defrule detectar-actividad-sospechosa
   (movimiento ?edificio)
   (horario ?edificio nocturno)
   =>
   (assert (actividad-sospechosa ?edificio))
   (print_facts))"""

r2 = """(defrule detectar-intrusion
   (actividad-sospechosa ?edificio)
   (puerta ?edificio abierta)
   =>
   (assert (posible-intrusion ?edificio))
    (print_facts))"""
    
r3 = """(defrule activar-seguridad
   (posible-intrusion ?edificio)
   =>
   (assert (alerta-seguridad ?edificio))
   (printout t "Alarma activada en edificio " ?edificio crlf)
    (print_facts))"""

r4 = """(defrule notificar-vigilante
   (alerta-seguridad ?edificio)
   =>
   (printout t "Vigilante notificado del edificio " ?edificio crlf)
    (halt))"""

"""
Compila y registra cada una de las reglas (r1, r2, r3, r4) en la Base de Conocimiento de CLIPS.
"""
env.build(r1)
env.build(r2)
env.build(r3)
env.build(r4)

"""
Reinicia la memoria de trabajo y vuelca nuevamente los hechos iniciales definidos en 'deffacts'.
Al entrar estos hechos, el algoritmo los evalúa al instante y coloca en la Agenda todas las 
coincidencias (activaciones) encontradas.
"""
env.reset()

"""
Llama a la función personalizada que imprime el estado actual de la Agenda.
Esto permite inspeccionar exactamente qué reglas se han activado y con qué hechos concretos
"""
print_agenda()

In [ ]:
"""
Ejecuta exactamente 1 paso del ciclo de inferencia (dispara la primera activación de la Agenda).
A diferencia de env.run() sin parámetros (que ejecuta todo hasta vaciar la Agenda), pasar el argumento '1' 
permite avanzar paso a paso (modo depuración), ejecutando únicamente la regla con mayor prioridad en este instante 
y actualizando la memoria de trabajo con los nuevos hechos que dicha regla haya inferido o modificado.
"""
env.run(1)
print_agenda()

In [ ]:
env.run(1)
print_agenda()

In [ ]:
env.run(1)
print_agenda()

In [ ]:
env.run()